In [2]:
import numpy as np
from tensorflow.keras.models import load_model
import tensorflow as tf

# === Simple eval for Model 1 (3-class: 0=Start,1=Off,2=Load) ===
# - Loads Router_test and RouterOCC_test, concatenates
# - FILTERS OUT all y==3 (Unknown), since M1 was trained without class 3
# - Evaluates model1_cnn on the filtered combined test set

# --- paths ---
MODEL_PATH        = "../../../models/Model1/model1_cnn.keras"
X_TEST_PATH       = "../../../data/test/Router_test_X.npy"
Y_TEST_PATH       = "../../../data/test/Router_test_y.npy"
X_TEST_PATH_OCC   = "../../../data/test/RouterOCC_test_X.npy"
Y_TEST_PATH_OCC   = "../../../data/test/RouterOCC_test_y.npy"

# --- re-declare the lambda function used in training ---
def deltas_fn(t):
    d = t[:, 1:, :] - t[:, :-1, :]
    zero = tf.zeros_like(d[:, :1, :])
    return tf.concat([zero, d], axis=1)

# --- load data ---
X1 = np.load(X_TEST_PATH).astype(np.float32)        # (N1,4,4)
y1 = np.load(Y_TEST_PATH).astype(np.int64)          # (N1,)

# OCC (may be all 3s). If files missing, skip gracefully.
try:
    X2 = np.load(X_TEST_PATH_OCC).astype(np.float32)
    y2 = np.load(Y_TEST_PATH_OCC).astype(np.int64)
except Exception:
    X2 = np.empty((0,4,4), dtype=np.float32)
    y2 = np.empty((0,), dtype=np.int64)

# combine
X_all = np.concatenate([X1, X2], axis=0)
y_all = np.concatenate([y1, y2], axis=0)

# --- FILTER OUT Unknown (label 3) to match M1 label space ---
mask = (y_all != 3)
X_test = X_all[mask]
y_test = y_all[mask]

if X_test.size == 0:
    raise ValueError("After removing Unknown (label=3), no samples remain to evaluate M1.")

# --- load & eval ---
model = load_model(MODEL_PATH, custom_objects={"deltas_fn": deltas_fn})
loss, acc = model.evaluate(X_test, y_test, verbose=1)
print(f"✅ M1 Test accuracy (Router [+ OCC filtered], classes 0/1/2 only): {acc:.4f}")

407/407 ━━━━━━━━━━━━━━━━━━━━ 1s 843us/step - accuracy: 1.0000 - loss: 1.0408e-04
✅ M1 Test accuracy (Router [+ OCC filtered], classes 0/1/2 only): 1.0000
